# Module 4: Tokenization and Data Formatting

In this final module, we bridge the gap between prepared tabular data and model-ready inputs. We'll learn how to use tokenizers, format chat histories, and create HuggingFace Dataset objects.

**Learning Outcome:** Participants can tokenize Thai-language utterances, format conversations into chat templates, and produce a HuggingFace Dataset ready for fine-tuning.

In [ ]:
%pip install transformers datasets torch pandas pyarrow ipywidgets

## Section 1: Loading the Checkpoint

We start by loading the consolidated dataset we exported in Module 3.

In [ ]:
import pandas as pd
from pathlib import Path

input_file = Path("outputs/consolidated_dataset.parquet")
df = pd.read_parquet(input_file)

print(f"Loaded {len(df)} records with columns: {df.columns.tolist()}")
df.head(2)

## Section 2: Tokenizer Basics

A **Tokenizer** converts text into a sequence of numbers (Token IDs). We'll use the tokenizer from the `Qwen2.5-0.5B-Instruct` model, which handles Thai text well.

In [ ]:
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

text = "สวัสดีค่ะ อยากเช็คยอดเงินค่ะ"
encoded = tokenizer.encode(text)

print(f"Text: {text}")
print(f"Token IDs: {encoded}")
print(f"Decoded back: {tokenizer.decode(encoded)}")

## Section 3: Padding and Truncation

When processing batches of messages, they must all have the same length. We use **Padding** for short messages and **Truncation** for long ones.

In [ ]:
batch_texts = ["สวัสดี", "อยากเช็คยอดเงินค้างชำระทั้งหมดในรอบบิลนี้"]

encoded_batch = tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Input IDs (notice the zeros for padding):\n", encoded_batch["input_ids"])
print("\nAttention Mask (1 for real token, 0 for padding):\n", encoded_batch["attention_mask"])

## Section 4: Chat Template Formatting

Modern models are trained to expect a specific format for conversations. We'll use the tokenizer's `apply_chat_template` to format our `session_history` and `user_message`.

In [ ]:
def format_for_chat(row):
    """Converts a record into the model's preferred chat format."""
    messages = []
    
    # Add turns from session_history (already dictionaries with 'role' and 'content')
    for turn in row['session_history']:
        messages.append(turn)
        
    # Add the current user message
    messages.append({"role": "user", "content": row['user_message']})
    
    # Convert to a single formatted string
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

df['formatted_chat'] = df.apply(format_for_chat, axis=1)

print("Sample formatted chat:")
print(df['formatted_chat'].iloc[1])

## Section 5: Label Encoding

Models need numeric labels for training. We'll map each unique intent string to a unique integer ID and add it as a `labels` column.

In [ ]:
# Create a mapping from intent to ID
intents = sorted(df['intent'].unique())
intent_to_id = {intent: i for i, intent in enumerate(intents)}

# Add the labels column
df['labels'] = df['intent'].map(intent_to_id)

print(f"Mapped {len(intents)} unique intents to numeric labels.")
print("Mapping example:", list(intent_to_id.items())[:3])
df[['intent', 'labels']].head()

## Section 6: Building a HuggingFace Dataset

We'll convert our pandas DataFrame into a HuggingFace `Dataset` and use the `.map()` function to tokenize all records efficiently.

In [ ]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    return tokenizer(examples["formatted_chat"], truncation=True, max_length=512)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

print("\nTokenized dataset features:", tokenized_dataset.features)
print("First record input_ids length:", len(tokenized_dataset[0]['input_ids']))

## Section 7: Train/Validation Split

We split the data into 80% training and 20% validation. We'll stratify the split by the `intent` column to ensure both sets have a similar label distribution.

In [ ]:
tokenized_dataset = tokenized_dataset.class_encode_column("intent")
final_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="intent")

print(final_split)

output_dir = Path("outputs/tokenized_dataset")
final_split.save_to_disk(output_dir)
print(f"\nFinal model-ready dataset saved to {output_dir}")